# Package

In [13]:
from pathlib import Path
import pandas as pd

# Import depuis ton utils.py
from utils import load_wide_from_feast, build_unrate_exog_dataset

from mlforecast import MLForecast
from sklearn.linear_model import Ridge

import numpy as np
from dateutil.relativedelta import relativedelta
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import Ridge
from mlforecast.utils import PredictionIntervals

# Importation

In [14]:
series_ids = [
    "BUSLOANS","CPIAUCSL","DPCERA3M086SBEA","INDPRO",
    "M2SL","OILPRICEX","RPI","SP500","TB3MS","UNRATE","USREC",
]

START = "1960-01-01"
END   = "2025-08-01"

In [15]:
# 1) WIDE dataset (tu veux l'appeler df)
df = load_wide_from_feast(
    "stationary_value:value",
    series_ids,
    start=START,
    end=END
)

# 2) Convert WIDE -> LONG (obligatoire pour build_unrate_exog_dataset)
df_stationary = (
    df.reset_index()
      .melt(id_vars="date", var_name="series_id", value_name="value")
)

# 3) Build target + exog + MLForecast format
df_model, ts_lr, exog_cols = build_unrate_exog_dataset(df_stationary)

print("df (wide) shape:", df.shape)
print("df_stationary (long) shape:", df_stationary.shape)
print("df_model shape:", df_model.shape)
print("ts_lr shape:", ts_lr.shape)
print("Exog cols:", exog_cols)

Using date as the event timestamp. To specify a column explicitly, please name it event_timestamp.
df (wide) shape: (788, 11)
df_stationary (long) shape: (8668, 3)
df_model shape: (788, 12)
ts_lr shape: (788, 13)
Exog cols: ['BUSLOANS', 'CPIAUCSL', 'DPCERA3M086SBEA', 'INDPRO', 'M2SL', 'OILPRICEX', 'RPI', 'SP500', 'TB3MS', 'USREC']


d:\Portofolio Data science\Time Series\Explainable_AI_Forecast_and_explain_the_Unemployment_of_USA\3_notebook\ML Experiment\utils.py:171: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  .dt.to_period("M")


# Dictionnaire

In [16]:

MLF_MODELS = {
    "RIDGE_EXOG_ONLY": lambda freq, *, alpha=1.0: MLForecast(
        models={"RIDGE": Ridge(alpha=float(alpha), random_state=0)},
        freq=freq,
        lags=[],               # EXOG-ONLY
        date_features=[],
    )
}

# Backteting

## Helpers dates & slicing

In [17]:
def _ensure_ms(x):
    x = pd.Timestamp(x)
    return x.to_period("M").to_timestamp(how="start").normalize()

def _n_windows_monthly(ds_start, ds_end):
    return (ds_end.year - ds_start.year) * 12 + (ds_end.month - ds_start.month) + 1

def _slice_cv_block(ts, cutoff_start, n_windows, h):
    """
    Construit une slice de données suffisante pour cross_validation sur un bloc :
    on garde toutes les obs jusqu'à cutoff_end + h (pour générer les ds prédites)
    """
    cutoff_end = cutoff_start + relativedelta(months=n_windows - 1)
    ds_end = cutoff_end + relativedelta(months=h)
    return ts[ts["ds"] <= ds_end].copy(), cutoff_end, ds_end

## Tuning Ridge (alpha) sur train

In [18]:
def _tune_alpha_on_train(ts_train, *, alpha_grid, cv=5):
    """
    Tuning alpha sur les données train (sklearn CV) — EXOG-ONLY.
    On entraîne sur toutes les lignes de ts_train (pas de fuite car <= cutoff_start du bloc).
    """
    drop_cols = {"unique_id", "ds", "y"}
    x_cols = [c for c in ts_train.columns if c not in drop_cols]
    X = ts_train[x_cols].values
    y = ts_train["y"].values

    grid = GridSearchCV(
        Ridge(fit_intercept=True, random_state=0),
        {"alpha": alpha_grid},
        scoring="neg_mean_absolute_error",
        cv=cv,
        n_jobs=-1,
    )
    grid.fit(X, y)
    return float(grid.best_estimator_.alpha), float(-grid.best_score_)

## Backteting

In [19]:
def run_backtesting_h12_monthly_tune_every_36m(
    ts,
    *,
    freq,
    h=12,
    exp_start="1990-01-01",
    exp_end="2025-08-01",
    step_size=1,
    pi_windows=24,
    levels=[95],
    # tuning
    tune_every_months=36,
    alpha_mode="cv",                # "cv" ou float
    alpha_grid=np.logspace(-4, 4, 30),
    min_train_n=None,               # optionnel
):
    ts = ts.copy()

    # --- dates MS propres ---
    ts["ds"] = (
        pd.to_datetime(ts["ds"], errors="coerce")
          .dt.to_period("M")
          .dt.to_timestamp(how="start")
          .dt.normalize()
    )
    if ts["ds"].isna().any():
        bad = ts[ts["ds"].isna()].head()
        raise ValueError(f"Dates 'ds' invalides après parsing. Exemples:\n{bad}")

    exp_start = _ensure_ms(exp_start)
    exp_end   = _ensure_ms(exp_end)

    cutoff_start_all = exp_start - relativedelta(months=h)
    cutoff_end_all   = exp_end   - relativedelta(months=h)
    total_partitions = _n_windows_monthly(cutoff_start_all, cutoff_end_all)

    # anti-fuite : on coupe à exp_end
    ts = ts[ts["ds"] <= exp_end].copy()

    # Conformal PI
    pi = PredictionIntervals(h=h, n_windows=pi_windows, method="conformal_distribution")

    # --- découpage en blocs de 36 mois (ou moins à la fin) ---
    blocks = []
    remaining = total_partitions
    cur_cutoff_start = cutoff_start_all

    while remaining > 0:
        n_win = min(tune_every_months, remaining)
        blocks.append((cur_cutoff_start, n_win))
        cur_cutoff_start = cur_cutoff_start + relativedelta(months=n_win)
        remaining -= n_win

    all_bkts = []
    alpha_history = []   # par bloc
    cv_mae_history = []  # par bloc

    for block_idx, (cutoff_start_blk, n_windows_blk) in enumerate(blocks, start=1):
        # --- data slice utile pour ce bloc ---
        ts_blk, cutoff_end_blk, ds_end_blk = _slice_cv_block(ts, cutoff_start_blk, n_windows_blk, h)

        # --- train pour tuner (<= cutoff_start_blk) ---
        ts_train_for_tune = ts[ts["ds"] <= cutoff_start_blk].copy()
        if min_train_n is not None and len(ts_train_for_tune) < int(min_train_n):
            continue

        # --- tuning alpha ---
        if alpha_mode == "cv":
            alpha_blk, cv_mae = _tune_alpha_on_train(ts_train_for_tune, alpha_grid=alpha_grid, cv=5)
        else:
            alpha_blk, cv_mae = float(alpha_mode), np.nan

        alpha_history.append(
            {"block": block_idx, "cutoff_start": cutoff_start_blk, "n_windows": n_windows_blk, "alpha": alpha_blk}
        )
        cv_mae_history.append(
            {"block": block_idx, "cutoff_start": cutoff_start_blk, "cv_mae": cv_mae}
        )

        # --- modèle Nixtla avec alpha figé sur le bloc ---
        mlf_blk = MLF_MODELS["RIDGE_EXOG_ONLY"](freq, alpha=alpha_blk)

        bkt_blk = mlf_blk.cross_validation(
            df=ts_blk,
            h=h,
            step_size=step_size,
            n_windows=n_windows_blk,
            prediction_intervals=pi,
            level=levels,
            fitted=True,
            static_features=[],
            dropna=True,
        )

        # tag bloc + alpha utilisé
        bkt_blk["tune_block"] = block_idx
        bkt_blk["alpha_used"] = alpha_blk
        all_bkts.append(bkt_blk)

    if not all_bkts:
        return pd.DataFrame(), {"error": "Aucun bloc backtest produit (min_train_n trop grand ?)"}

    bkt_df = pd.concat(all_bkts, ignore_index=True)

    # garder uniquement la plage d’expérience
    bkt_df = bkt_df[(bkt_df["ds"] >= exp_start) & (bkt_df["ds"] <= exp_end)].copy()
    bkt_df = bkt_df.sort_values(["unique_id", "ds", "cutoff"]).reset_index(drop=True)

    meta = {
        "h": h,
        "step_size": step_size,
        "exp_start": exp_start,
        "exp_end": exp_end,
        "cutoff_start": cutoff_start_all,
        "cutoff_end": cutoff_end_all,
        "partitions": total_partitions,
        "pi_windows": pi_windows,
        "tune_every_months": tune_every_months,
        "alpha_mode": alpha_mode,
        "alpha_grid": list(alpha_grid) if alpha_mode == "cv" else None,
        "alpha_history": alpha_history,
        "cv_mae_history": cv_mae_history,
    }

    return bkt_df, meta

### Execution

### Import + config

In [20]:
PROJECT_ROOT = Path.cwd().parent
print("PROJECT_ROOT =", PROJECT_ROOT.resolve())

FREQ = "MS"

H = 12
STEP_SIZE = 1
PI_WINDOWS = 3
LEVELS = [95]

EXP_START = "1990-01-01"
EXP_END   = "2025-08-01"

TUNE_EVERY_MONTHS = 36
ALPHA_MODE = "cv"  # ou float (ex: 1.0)
ALPHA_GRID = np.logspace(-4, 4, 30)

PROJECT_ROOT = D:\Portofolio Data science\Time Series\Explainable_AI_Forecast_and_explain_the_Unemployment_of_USA\3_notebook


# Anti_fuite

In [21]:
ts_ridge = ts_lr.copy()

ts_ridge["ds"] = (
    pd.to_datetime(ts_ridge["ds"], errors="coerce")
      .dt.to_period("M")
      .dt.to_timestamp(how="start")
      .dt.normalize()
)

# anti-fuite
ts_ridge = ts_ridge[ts_ridge["ds"] <= pd.Timestamp(EXP_END)].copy()

print("ts_ridge shape:", ts_ridge.shape)
ts_ridge.head()

ts_ridge shape: (788, 13)


,unique_id,ds,y,BUSLOANS,CPIAUCSL,DPCERA3M086SBEA,INDPRO,M2SL,OILPRICEX,RPI,SP500,TB3MS,USREC
0,UNRATE,1960-01-01,-0.8,0.011578,-0.006156,0.001204,0.091976,0.001323,0.0,0.020977,0.017909,0.30,0.0
1,UNRATE,1960-02-01,-1.1,0.011905,-0.003767,0.006009,0.076960,0.002007,0.0,0.014565,-0.025663,-0.19,0.0
2,UNRATE,1960-03-01,-0.2,-0.008356,-0.005455,0.021240,0.007959,0.001324,0.0,0.006250,-0.070857,-1.18,0.0
3,UNRATE,1960-04-01,0.0,-0.009098,0.005090,0.033752,-0.025916,0.000634,0.0,0.006489,-0.040442,-1.12,0.0
4,UNRATE,1960-05-01,0.0,-0.000359,0.003383,0.009040,-0.018119,0.003977,0.0,0.007747,-0.010090,-0.67,1.0


# Run Backteting

In [22]:
bkt_ridge, meta_ridge = run_backtesting_h12_monthly_tune_every_36m(
    ts=ts_ridge,
    freq=FREQ,
    h=H,
    exp_start=EXP_START,
    exp_end=EXP_END,
    step_size=STEP_SIZE,
    pi_windows=PI_WINDOWS,
    levels=LEVELS,
    tune_every_months=TUNE_EVERY_MONTHS,
    alpha_mode=ALPHA_MODE,
    alpha_grid=ALPHA_GRID,
    min_train_n=36,
)

bkt_ridge_final = (
    bkt_ridge.sort_values(["unique_id", "ds", "cutoff"])
             .groupby(["unique_id", "ds"], as_index=False)
             .tail(1)
             .reset_index(drop=True)
)


print("✅ meta_ridge keys:", list(meta_ridge.keys()))
print("bkt_ridge rows:", len(bkt_ridge))
print("bkt_ridge_final rows        :", len(bkt_ridge_final))
print("duplicates (unique_id, ds)  :", bkt_ridge_final.duplicated(["unique_id","ds"]).sum())
print("alphas used (unique)        :", sorted(bkt_ridge_final["alpha_used"].unique().tolist()))

bkt_ridge_final.head()

✅ meta_ridge keys: ['h', 'step_size', 'exp_start', 'exp_end', 'cutoff_start', 'cutoff_end', 'partitions', 'pi_windows', 'tune_every_months', 'alpha_mode', 'alpha_grid', 'alpha_history', 'cv_mae_history']
bkt_ridge rows: 5070
bkt_ridge_final rows        : 428
duplicates (unique_id, ds)  : 0
alphas used (unique)        : [0.03039195382313198, 0.05736152510448681, 0.38566204211634725, 1.3738237958832638, 2.592943797404667, 4.893900918477489, 9.236708571873866]


,unique_id,ds,cutoff,y,RIDGE,RIDGE-lo-95,RIDGE-hi-95,tune_block,alpha_used
0,UNRATE,1990-01-01,1989-12-01,0.0,-0.374110,-0.607142,-0.141079,1,0.385662
1,UNRATE,1990-02-01,1990-01-01,0.1,-0.415840,-1.277387,0.445707,1,0.385662
2,UNRATE,1990-03-01,1990-02-01,0.2,-0.424608,-1.467775,0.618558,1,0.385662
3,UNRATE,1990-04-01,1990-03-01,0.2,-0.334295,-1.268968,0.600377,1,0.385662
4,UNRATE,1990-05-01,1990-04-01,0.2,-0.069912,-0.794322,0.654498,1,0.385662


# Tableau

In [25]:
import pandas as pd

alpha_hist = pd.DataFrame(meta_ridge["alpha_history"]).copy()

alpha_hist["cutoff_start"] = pd.to_datetime(alpha_hist["cutoff_start"])
alpha_hist["cutoff_end"] = alpha_hist["cutoff_start"] + pd.offsets.MonthBegin(1) * (alpha_hist["n_windows"] - 1)

alpha_hist = alpha_hist[["block", "cutoff_start", "cutoff_end", "n_windows", "alpha"]]
alpha_hist

C:\Users\Mita\AppData\Local\Temp\ipykernel_16024\295711611.py:6: PerformanceWarning: Adding/subtracting object-dtype array to DatetimeArray not vectorized.
  alpha_hist["cutoff_end"] = alpha_hist["cutoff_start"] + pd.offsets.MonthBegin(1) * (alpha_hist["n_windows"] - 1)


,block,cutoff_start,cutoff_end,n_windows,alpha
0,1,1989-01-01,1991-12-01 00:00:00,36,0.385662
1,2,1992-01-01,1994-12-01 00:00:00,36,1.373824
2,3,1995-01-01,1997-12-01 00:00:00,36,0.385662
3,4,1998-01-01,2000-12-01 00:00:00,36,0.057362
4,5,2001-01-01,2003-12-01 00:00:00,36,2.592944
5,6,2004-01-01,2006-12-01 00:00:00,36,4.893901
6,7,2007-01-01,2009-12-01 00:00:00,36,9.236709
7,8,2010-01-01,2012-12-01 00:00:00,36,0.057362
8,9,2013-01-01,2015-12-01 00:00:00,36,0.030392
9,10,2016-01-01,2018-12-01 00:00:00,36,4.893901


L’évolution du paramètre alpha du modèle Ridge montre que le niveau de régularisation change selon les périodes économiques. Pendant les phases plus stables, comme la fin des années 1990 ou la période 2010–2015, les valeurs d’alpha restent faibles. Cela signifie que le modèle utilise davantage les variables exogènes et capture plus facilement les relations entre les indicateurs macroéconomiques et le chômage.

À l’inverse, pendant les périodes d’instabilité économique ou de rupture, comme la crise de 2008 ou la période post-COVID, les valeurs d’alpha augmentent. Le modèle applique alors une régularisation plus forte pour rester stable et éviter le sur-apprentissage face à des relations plus incertaines.

Globalement, ces résultats montrent que les liens entre les variables explicatives et le chômage changent dans le temps. Le tuning régulier du paramètre alpha permet donc d’adapter le modèle aux différents régimes économiques et d’améliorer la robustesse des prévisions.

# Graphique démontrant l'évolution de RIDGE dan le temps

In [ ]:
from pathlib import Path
EXPORT_DIR = Path("outputs/streamlit")
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
bkt_ridge_final.to_parquet(EXPORT_DIR / "bkt_ridge_final.parquet", index=False)

In [41]:
# ============================================================
# Plot unique : RIDGE seul (format bkt_ridge_final déjà prêt)
# Colonnes dispo:
# ['unique_id','ds','y','RIDGE','RIDGE-lo-95','RIDGE-hi-95', ...]
# ============================================================

import pandas as pd
from utilsforecast.plotting import plot_series

# 1) Copier + typer ds
d = bkt_ridge_final.copy()
d["ds"] = pd.to_datetime(d["ds"])
d = d.sort_values(["unique_id", "ds"])

# 2) Construire df_obs et df_fcst (format utilsforecast)
df_obs = d[["unique_id", "ds", "y"]].drop_duplicates(["unique_id", "ds"]).copy()

df_fcst = d[[
    "unique_id", "ds",
    "RIDGE", "RIDGE-lo-95", "RIDGE-hi-95"
]].drop_duplicates(["unique_id", "ds"]).copy()

# 3) Plot
fig = plot_series(
    df=df_obs,
    forecasts_df=df_fcst,
    level=[95],
    engine="plotly",
).update_layout(height=480)

# 4) Renommer légendes proprement
for trace in fig.data:
    n = (trace.name or "")
    nl = n.lower()

    if n == "y":
        trace.name = "Unemployment rate (%)"
    elif n == "RIDGE":
        trace.name = "Ridge Regression (exog)"
    elif "level_95" in nl and "ridge" in nl:
        trace.name = "Ridge 95% Prediction Interval"
    elif "level_95" in nl:
        trace.name = "95% Prediction Interval"

fig.show()

[2026-02-17 20:57:46,963] ERROR in app: Exception on /_dash-update-component [POST]
Traceback (most recent call last):
  File "d:\Portofolio Data science\Time Series\Explainable_AI_Forecast_and_explain_the_Unemployment_of_USA\venv\Lib\site-packages\dash\dash.py", line 1492, in _prepare_callback
    cb = self.callback_map[output]
         ~~~~~~~~~~~~~~~~~^^^^^^^^
KeyError: '..g_main.figure...g_c.figure...metrics.children...slider.value...pos.children...state.data..'

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "d:\Portofolio Data science\Time Series\Explainable_AI_Forecast_and_explain_the_Unemployment_of_USA\venv\Lib\site-packages\flask\app.py", line 917, in full_dispatch_request
    rv = self.dispatch_request()
         ^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\Portofolio Data science\Time Series\Explainable_AI_Forecast_and_explain_the_Unemployment_of_USA\venv\Lib\site-packages\flask\app.py", line 902, in dispatch_request
  